# Обучение ML модели для Bybit Trader в Google Colab

Этот ноутбук обучает LSTM модель на исторических данных криптовалют.

In [ ]:
# Установка зависимостей
!pip install -q tensorflow scikit-learn pandas numpy joblib

In [ ]:
# Загрузка данных с сервера
# Вариант 1: Через SCP (нужен SSH доступ)
# !scp -r root@88.210.10.145:/root/Bybit_Trader/ml_data/*.csv ./

# Вариант 2: Загрузить файлы вручную через интерфейс Colab
from google.colab import files
print("Загрузите все CSV файлы из ml_data/")
uploaded = files.upload()

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import joblib

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
def load_all_data():
    """Загрузить все CSV файлы"""
    print(f"\n📂 Loading data...")
    
    all_data = []
    symbols = []
    
    for filename in os.listdir('.'):
        if filename.endswith('.csv') and filename.startswith('klines_'):
            symbol = filename.split('_')[1]  # BTCUSDT
            
            print(f"   Loading {filename}...")
            df = pd.read_csv(filename)
            df['symbol'] = symbol
            all_data.append(df)
            symbols.append(symbol)
            print(f"      {len(df)} records")
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    print(f"\n✅ Loaded {len(combined_df)} total records from {len(symbols)} symbols")
    print(f"   Symbols: {', '.join(symbols)}")
    
    return combined_df, symbols

df, symbols = load_all_data()

In [ ]:
def prepare_features(df):
    """Подготовить фичи для обучения"""
    print(f"\n🔧 Preparing features...")
    
    feature_cols = [
        'open', 'high', 'low', 'close', 'volume',
        'rsi', 'macd', 'macd_signal', 'bb_upper', 'bb_middle', 'bb_lower',
        'atr', 'stoch_k', 'stoch_d', 'sma_20', 'sma_50', 'ema_12', 'ema_26',
        'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos'
    ]
    
    df_clean = df.dropna(subset=feature_cols + ['close'])
    print(f"   Removed {len(df) - len(df_clean)} rows with NaN")
    
    X = df_clean[feature_cols].values
    y = df_clean['close'].values.reshape(-1, 1)
    
    print(f"✅ Features prepared: {X.shape}")
    
    return X, y

X, y = prepare_features(df)

In [ ]:
def create_sequences(X, y, sequence_length=60):
    """Создать последовательности для LSTM"""
    print(f"\n🔄 Creating sequences (length={sequence_length})...")
    
    X_seq, y_seq = [], []
    
    for i in range(len(X) - sequence_length):
        X_seq.append(X[i:i + sequence_length])
        y_seq.append(y[i + sequence_length])
    
    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    
    print(f"✅ Created {len(X_seq)} sequences")
    print(f"   X shape: {X_seq.shape}")
    print(f"   y shape: {y_seq.shape}")
    
    return X_seq, y_seq

# Нормализация
print(f"\n📏 Normalizing data...")
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

print(f"✅ Data normalized")

# Создаём последовательности
SEQUENCE_LENGTH = 60
X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQUENCE_LENGTH)

In [ ]:
# Train/Test split
split_idx = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

print(f"\n📊 Train/Test split:")
print(f"   Train: {len(X_train)} samples")
print(f"   Test:  {len(X_test)} samples")

In [ ]:
# Строим модель
print(f"\n🏗️  Building model...")

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, X.shape[1])),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print(f"✅ Model built")
model.summary()

In [ ]:
# Обучение
print(f"\n🎓 Training model...")

EPOCHS = 100
BATCH_SIZE = 32

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Оценка
print(f"\n📈 Evaluating model...")
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print(f"   Train Loss: {train_loss:.6f}, MAE: {train_mae:.6f}")
print(f"   Test Loss:  {test_loss:.6f}, MAE: {test_mae:.6f}")

In [ ]:
# Сохранение
print(f"\n💾 Saving model...")

model.save('bybit_lstm_model.h5')
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

# Метаданные
import json
metadata = {
    'version': datetime.now().strftime("%Y%m%d_%H%M%S"),
    'symbols': symbols,
    'total_samples': len(X_seq),
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'sequence_length': SEQUENCE_LENGTH,
    'features': X.shape[1],
    'epochs_trained': len(history.history['loss']),
    'train_loss': float(train_loss),
    'train_mae': float(train_mae),
    'test_loss': float(test_loss),
    'test_mae': float(test_mae)
}

with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Model saved!")
print(f"\nDownload these files:")
print(f"  - bybit_lstm_model.h5")
print(f"  - scaler_X.pkl")
print(f"  - scaler_y.pkl")
print(f"  - metadata.json")

In [ ]:
# Скачать файлы
from google.colab import files

files.download('bybit_lstm_model.h5')
files.download('scaler_X.pkl')
files.download('scaler_y.pkl')
files.download('metadata.json')

print("\n✅ Обучение завершено! Загрузи файлы на сервер в /data/ml_models/")